In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score


import gc

In [ ]:
data = np.load("user_pca_reduced_and_counts.npz", allow_pickle=True)

In [ ]:
user_pca_reduced = data["user_pca_reduced"].item()

In [ ]:
len(user_pca_reduced)

1737004

In [ ]:
embeddings_raw = np.stack(list(user_pca_reduced.values()))
usernames = list(user_pca_reduced.keys())

In [ ]:
pca_dims= [20,30,40,50]

In [ ]:
min_frac = 0.005

In [ ]:
ks= [10,15,20,25,30,35,40,45,50,55,60]

In [ ]:
results = []

for pca_dim in pca_dims:
    embeddings = embeddings_raw[:, :pca_dim]

    for k in ks:
        print(k)
        mbk = MiniBatchKMeans(
            n_clusters=k,
            batch_size=100000,
            n_init=3,
            random_state=42
        )
        cluster_ids = mbk.fit_predict(embeddings)

        inertia = mbk.inertia_
        counts = np.bincount(cluster_ids, minlength=k)
        cluster_frac = counts / counts.sum()
        max_frac = cluster_frac.max()
        herfindahl = np.sum(cluster_frac**2)
        n_small = (cluster_frac < min_frac).sum()
        frac_small_total = cluster_frac[cluster_frac < min_frac].sum()

        # Silhouette Subsample
        sample_idx = np.random.choice(len(cluster_ids), size=100000, replace=False)
        sil_score = silhouette_score(embeddings[sample_idx], cluster_ids[sample_idx])

        results.append({
            'dim': pca_dim,
            'k': k,
            'inertia': inertia,
            'cluster_max_frac': max_frac,
            "herfindahl": herfindahl,
            "n_small_clusters": int(n_small),
            "small_clusters_frac_total": float(frac_small_total),
            'silhouette': sil_score
        })

25
30
35
40
45
50
55
60


In [ ]:
df = pd.DataFrame(results)
df.to_csv(f"clustering_results_{pca_dims[0]}-{pca_dims[-1]}.csv", index=False)